# Chapter 2. 멀티암 밴딧 — ε-greedy 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter02_1_epsilon_greedy.ipynb)

책 본문: [2.1 탐험과 활용: ε-greedy](https://smhanlab.com/book-ml/kor/ml2/chapter02/1.html)

이 노트북은 \(\varepsilon\)-greedy 밴딧 알고리즘을 직접 실행합니다.
책에서 한 문장씩 등장하는 세 가지 주장을 숫자로 확인합니다:

1. **\(\varepsilon=0\)(순수 탐욕)은 "한 번의 나쁜 운"에 최적 팔을 영원히 잃는다**
   (같은 시드에서 \(\varepsilon=0\) vs 0.1을 비교).
2. **탐험에도 비용이 있다** — \(\varepsilon\)를 0→1로 훑으면 총보상이
   양쪽 극단에서 꺾이고, 최대치가 가운데(\(\varepsilon \approx 0.05\))에 있다.
3. **\(\varepsilon=0\)의 실패는 "가끔"이 아니라 거의 매번 일어난다** —
   200번 실험의 총보상 분포가 거의 3000(= 2000스텝 × 평균 1.5) 아래에 몰려 있다.

## 1. 공통 설정

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import random

IMG = "/home/smhan/book-ml/kor/src/images"

def run_bandit(true_means, epsilon, steps, seed=0):
    """책 2.1절의 epsilon_greedy_bandit + 총보상/당김횟수 반환."""
    random.seed(seed)
    k = len(true_means)
    Q, N, total = [0.0]*k, [0]*k, 0.0
    for t in range(steps):
        a = random.randrange(k) if random.random() < epsilon \
            else max(range(k), key=lambda i: Q[i])   # 활용(동점 시 첫 팔)
        r = random.gauss(true_means[a], 1.0)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]                     # 증분 갱신(온라인 평균)
        total += r
    return Q, N, total

## 2. \(\varepsilon=0\): 한 번의 나쁜 운이 만드는 '영원한 틀린 확신'

책의 "자주 하는 실수" 섹션과 정확히 같은 실험입니다. 진짜 평균이
\([2.0, 1.0]\)(**A가 진짜 최선**)인 2팔 밴딧에서, \(random.seed(11)\)로
\(200\)스텝을 돌립니다. A의 첫 관찰 보상이 \(-0.113\)로 잡음 때문에 낮게
나온 이 시드에서, \(\varepsilon=0\)과 \(\varepsilon=0.1\)의 운명이
완전히 갈립니다.

In [2]:
Q0, N0, tot0 = run_bandit([2.0, 1.0], 0.0, 200, seed=11)
Q1, N1, tot1 = run_bandit([2.0, 1.0], 0.1, 200, seed=11)
print(f"eps=0.0  Q = {[round(x,3) for x in Q0]},  N = {N0},  총보상 = {tot0:.1f}")
print(f"eps=0.1  Q = {[round(x,3) for x in Q1]},  N = {N1},  총보상 = {tot1:.1f}")

# 책 본문에서 인용한 숫자와 정확히 일치하는지 확인
assert N0 == [1, 199] and abs(Q0[0] - (-0.113)) < 2e-3 and abs(Q0[1] - 0.894) < 2e-3
assert N1 == [183, 17] and abs(Q1[0] - 1.974) < 2e-3 and abs(Q1[1] - 0.739) < 2e-3
print("책의 숫자와 일치: eps=0은 A를 딱 1번만 당기고(B만 199회), eps=0.1은 진짜 최선 A를 183회 당김.")

eps=0.0  Q = [-0.113, 0.894],  N = [1, 199],  총보상 = 177.8
eps=0.1  Q = [1.974, 0.739],  N = [183, 17],  총보상 = 373.7
책의 숫자와 일치: eps=0은 A를 딱 1번만 당기고(B만 199회), eps=0.1은 진짜 최선 A를 183회 당김.


## 3. \(\varepsilon\)를 훑으면: 탐험에도 비용이 있다

표준 3팔 밴딧 \(q^* = [1.0, 1.5, 2.0]\), \(T = 2000\)스텝. 각
\(\varepsilon\)마다 **200번 독립 실험**(시드 1000~1199)을 돌려 평균을
냅니다. 오라클(매 스텝 진짜 최선 팔)의 총보상은 \(2000 \times 2.0 = 4000\).
후회 \(\mathcal{R}_{2000} = 4000 - \)총보상입니다.

In [3]:
true_means = [1.0, 1.5, 2.0]
steps = 2000
oracle = steps * max(true_means)
epsilons = [0.0, 0.01, 0.05, 0.1, 0.2, 0.5]

mean_tots, tot_lists = [], {}
for eps in epsilons:
    tots, found = [], 0
    for trial in range(200):
        Q, N, tot = run_bandit(true_means, eps, steps, seed=1000 + trial)
        tots.append(tot)
        if N[2] >= 0.5 * steps:      # 최선 팔(C)이 50% 이상: '찾아낸' 실험
            found += 1
    mean_tot = sum(tots) / len(tots)
    mean_tots.append(mean_tot)
    tot_lists[eps] = tots
    print(f"eps={eps:<5}  평균총보상={mean_tot:8.1f}  평균후회={oracle-mean_tot:8.1f}  "
          f"최선찾은실험={found}/200")

eps=0.0    평균총보상=  2245.6  평균후회=  1754.4  최선찾은실험=5/200


eps=0.01   평균총보상=  3657.1  평균후회=   342.9  최선찾은실험=169/200
eps=0.05   평균총보상=  3872.4  평균후회=   127.6  최선찾은실험=197/200


eps=0.1    평균총보상=  3862.8  평균후회=   137.2  최선찾은실험=200/200


eps=0.2    평균총보상=  3785.9  평균후회=   214.1  최선찾은실험=200/200
eps=0.5    평균총보상=  3498.1  평균후회=   501.9  최선찾은실험=200/200


## 4. \(\varepsilon\) 곡선과, \(\varepsilon=0\) 실패의 얼굴

왼쪽: 평균 총보상 곡선 — 양극(0과 1 근처)에서 꺾이고 **가운데(0.05)가 최대**.
오른쪽: 평균 후회 — 같은 모양의 'V'가 뒤집힌 것. \(\varepsilon=0\)의 후회
\(1754\)는 "단 한 번의 나쁜 운"이 2000스텝 총보상의 절반 가까이를 만들었다는 뜻.
아래: \(\varepsilon=0\) 200번 실험의 총보상 분포 — 거의 전부 3000 아래(97%가
최선 팔을 못 찾음). \(\varepsilon=0\)가 *가끔*은 잘 되는 것처럼 보이는
우측 꼬리(운이 좋게 첫 팔이 최선이었던 3%)도 함께 보입니다.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax1, ax2 = axes
ax1.plot(epsilons, mean_tots, "o-", color="#1f77b4", lw=2, ms=7)
ax1.axhline(oracle, color="gray", ls="--", lw=1, label=f"Oracle {oracle:.0f}")
ax1.set_xlabel("ε"); ax1.set_ylabel("Mean total reward (2000 steps)")
ax1.set_title("ε-greedy: sweeping ε\ntotal reward dips at both extremes")
ax1.legend(); ax1.grid(alpha=0.3)

mean_regs = [oracle - m for m in mean_tots]
ax2.plot(epsilons, mean_regs, "o-", color="#d62728", lw=2, ms=7)
ax2.set_xlabel("ε"); ax2.set_ylabel("Mean regret R_T")
ax2.set_title("Regret = Oracle − total reward\nminimum in the middle (ε≈0.05)")
ax2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch02_1_epsilon_tradeoff.svg", bbox_inches="tight")
plt.show()

tot0 = tot_lists[0.0]
below3000 = sum(1 for x in tot0 if x < 3000)
fig2, ax = plt.subplots(figsize=(7, 4.2))
ax.hist(tot0, bins=25, color="#d62728", alpha=0.8, edgecolor="white")
ax.axvline(3000, color="k", ls="--", lw=1, label="3000 (2000 steps × mean 1.5)")
ax.set_xlabel("Total reward (2000 steps)")
ax.set_ylabel("Number of experiments (out of 200)")
ax.set_title(f"Total reward distribution for ε=0 (pure greedy)\n{below3000}/200 ({below3000/2:.0f}%) below 3000")
ax.legend(); ax.grid(alpha=0.3)
fig2.tight_layout()
fig2.savefig(IMG + "/ch02_1_greedy_zero_hist.svg", bbox_inches="tight")
print(f"eps=0 총보상: min={min(tot0):.1f}, max={max(tot0):.1f}, 3000 미만 {below3000}/200")
plt.show()

eps=0 총보상: min=1849.6, max=3997.7, 3000 미만 177/200
